# ⏳ Notebook 2: Time-Bounded Leases

A **lease** is a lock with an expiration time. The holder must *renew* the lease before it expires; otherwise it's automatically released and someone else can grab it. Used by Chubby, etcd, ZooKeeper, Kubernetes leader election.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/lease
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟨 BETTER: a leased lock

In [1]:
import time
from dataclasses import dataclass
from typing import Optional

@dataclass
class Lease:
    holder: Optional[str] = None
    expires_at: float = 0.0

    def _expired(self):
        return time.monotonic() >= self.expires_at

    def acquire(self, who: str, ttl: float) -> bool:
        if self.holder is None or self._expired():
            self.holder = who
            self.expires_at = time.monotonic() + ttl
            return True
        return False

    def renew(self, who: str, ttl: float) -> bool:
        if self.holder == who and not self._expired():
            self.expires_at = time.monotonic() + ttl
            return True
        return False

    def release(self, who: str):
        if self.holder == who:
            self.holder = None
            self.expires_at = 0.0


## ⚡ Scenario: holder crashes, lease expires, someone else takes over

In [2]:
lease = Lease()
print('A acquires (ttl=1s):', lease.acquire('A', ttl=1.0))
print('B tries immediately:', lease.acquire('B', ttl=1.0))
time.sleep(1.1)  # A 'crashed' and never renewed
print('B tries after expiry:', lease.acquire('B', ttl=1.0))
print('holder now:', lease.holder)


A acquires (ttl=1s): True
B tries immediately: False


B tries after expiry: True
holder now: B


## 🔄 Healthy holder renews before expiry

In [3]:
lease = Lease()
lease.acquire('A', ttl=0.5)
for i in range(3):
    time.sleep(0.3)
    ok = lease.renew('A', ttl=0.5)
    print(f'  renew #{i}: {ok}, holder={lease.holder}')


  renew #0: True, holder=A


  renew #1: True, holder=A


  renew #2: True, holder=A


## 🧠 Comparison

| | Plain lock | Lease |
|---|---|---|
| Holder crashes | locked forever | auto-released after TTL |
| Cost while healthy | none | periodic renew RPC |
| Tunable safety | n/a | shorter TTL = faster recovery, more renew traffic |

## ⚠️ Gotchas

- **Clock skew** between nodes can let a 'live' holder think it still has the lease while a 'fresh' holder also thinks it does. → see the *split-brain-and-fencing* lab for **fencing tokens** that fix this.
- TTL should be **>> max network round-trip + GC pause** to avoid spurious expirations.
